In [5]:
import torch, time
from functorch import vmap

device = torch.device("cuda")

# Create data directly on GPU
X = torch.randn(50000, 512, device=device)

def f(x):
    return torch.tanh(torch.sin(x) * 2.5)

# Warm-up GPU
for _ in range(10):
    f(X[0])

# Baseline: Python loop (very slow on GPU)
start = time.time()
Y_loop = torch.stack([f(x) for x in X])
loop_time = time.time() - start

torch.cuda.synchronize()

# vmap version
start = time.time()
Y_vmap = vmap(f)(X)
# Y_vmap = f(X)
torch.cuda.synchronize()
vmap_time = time.time() - start

print(f"Python loop: {loop_time:.4f}s")
print(f"vmap:        {vmap_time:.4f}s")
print("Match:", torch.allclose(Y_loop, Y_vmap, atol=1e-6))


Python loop: 1.8708s
vmap:        0.0043s
Match: True


/tmp/ipykernel_21128/858934200.py:25: FutureWarning: We've integrated functorch into PyTorch. As the final step of the integration, `functorch.vmap` is deprecated as of PyTorch 2.0 and will be deleted in a future version of PyTorch >= 2.3. Please use `torch.vmap` instead; see the PyTorch 2.0 release notes and/or the `torch.func` migration guide for more details https://pytorch.org/docs/main/func.migrating.html
  Y_vmap = vmap(f)(X)
